# ZelusBench: Dimensionality — 2D**Flat plane (x, y). Simpler coordinate tracking.**Fixed dimensionality, randomized background conditions(chain depth, DAG structure, distractors, transforms).Backgrounds are deterministic per scenario index — shared across all levels.- Dimension: 2D- 15 scenarios

In [ ]:
import kaggle_benchmarks as kbench
import numpy as np
import random
import re
import time

from zelusbench.scenarios.config import (
    ScenarioConfig, DAGStructure, QueryType,
    DistractorLevel, TransformDensity,
)
from zelusbench.scenarios.generator import ScenarioGenerator
from zelusbench.evaluation.scorer import score_query, score_response




DIM = 2
SEEDS = 15

print(f"ZelusBench Dimensionality — {DIM}D")
print(f"Dim: {DIM} | Seeds: {SEEDS}")

In [ ]:
@kbench.task(name="zelusbench_attn_dim_2")
def zelusbench_attn_dim_2(llm) -> tuple[float, float]:
    _start = time.time()
    print(f"Running {SEEDS} scenarios (dim={DIM}D)...")
    print("=" * 60)

    all_scores = []

    for i in range(SEEDS):
        bg_rng = random.Random(i * 7919)
        cfg = ScenarioConfig.randomize_except(bg_rng, pinned={
            "dim": DIM,
            "num_queries": 3,
            "seed": DIM * 1000 + i,
        })
        gen = ScenarioGenerator(cfg)
        scenario = gen.generate(f"dim_{DIM}d_{i}")

        response = llm.prompt(scenario.prompt)
        scores = score_response(response, scenario)
        all_scores.extend(scores)

        avg = float(np.mean([s.score for s in scores]))
        q_depths = [s.chain_depth for s in scores]
        tiers = [s.tier.name for s in scores]
        bg = f"dag={cfg.dag_structure.name} depth={cfg.min_chain_depth}-{cfg.max_chain_depth} dist={cfg.distractor_level.name} tx={cfg.transform_density.name}"
        print(f"  [{i+1}/{SEEDS}] avg={avg:.2f} q_depths={q_depths} tiers={tiers} | {bg}")

    overall = float(np.mean([s.score for s in all_scores]))
    std = float(np.std([s.score for s in all_scores]))
    elapsed = time.time() - _start
    print(f"\nOverall: {overall:.3f} +/- {std:.3f} | {len(all_scores)} queries | {elapsed:.1f}s")
    kbench.assertions.assert_true(overall >= 0, expectation=f"{DIM}D valid (got {overall:.3f})")
    return overall, std


zelusbench_attn_dim_2.run(llm=kbench.llm)

In [ ]:
%choose zelusbench_attn_dim_2